In [1]:
from interfacemaster.twinning_search import search_low_index_twinning
from interfacemaster.twinning_jobflow import TwinningJobflowMaker
from pymatgen.core.structure import Structure

stct = Structure.from_file('LNO_prim.cif').get_primitive_structure()
# 1. 先搜索孪晶候选
search_results = search_low_index_twinning(parent = stct, 
                                           child = stct, 
                                           max_sigma=20, #最大sigma
                                           ortho_only=True, #要求正交超胞
                                          max_strain=5e-2,#最大strain（对于立方晶系直接设为很小）
                                          max_atoms=200, #最大原子数
                                           slab_length = 10, #slab厚度
                                          require_equivalent_terminations = True, #要求两个晶界端面等价
                                            termination_ftol=0.15, #端面分类精度
                                            termination_tol=0.15, #端面分类精度
                                          max_results = 25, #搜索到的符合要求的最大匹配数量
                                          prefilter_limit = 100, #搜索匹配的数量,
                                           debug_filters=True,
                                          )

/opt/anaconda3/envs/3.12/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/anaconda3/envs/3.12/lib/python3.12/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: dlopen(/opt/anaconda3/envs/3.12/lib/python3.12/site-packages/torch_scatter/_scatter_cpu.so, 0x0006): Symbol not found: __ZN2at4_ops16div__Tensor_mode4callERNS_6TensorERKS2_NSt3__18optionalIN3c1017basic_string_viewIcEEEE
  Referenced from: <EFDA55F0-CA14-39BB-A469-C12A0718A392> /opt/anaconda3/envs/3.12/lib/python3.12/site-packages/torch_scatter/_scatter_cpu.so
  Expected in:     <DA215AD3-6EAE-3755-B6A5-A8EB4EF952B0> /opt/anaconda3/envs/3.12/lib/python3.12/site-packages/torch/lib/libtorch_cpu.dylib
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/opt/anaconda3/envs/3.12/lib/pyt

--- 增强深度搜索开始 (Max Sigma: 20, HKL Limit: 4) ---
正在进行正交性过滤 (tol_ortho=0.01)...
正在进行原子数过滤 (max_atoms=200)...
  - atoms_est hkl [-1 -1 -2] sigma 2 -> 128 (rep_k=2, h_single=8.223Å, slab_length=10)
  - atoms_est hkl [-1 -1 -2] sigma 2 -> 64 (rep_k=2, h_single=8.223Å, slab_length=10)
  - atoms_est hkl [-1 -1 -2] sigma 2 -> 128 (rep_k=2, h_single=8.223Å, slab_length=10)
  - atoms_est hkl [-1 -1  0] sigma 2 -> 96 (rep_k=1, h_single=14.218Å, slab_length=10)
  - atoms_est hkl [-1 -1  0] sigma 2 -> 96 (rep_k=1, h_single=14.218Å, slab_length=10)
  - atoms_est hkl [-1 -1  2] sigma 2 -> 64 (rep_k=2, h_single=5.831Å, slab_length=10)
  - atoms_est hkl [-1  0 -3] sigma 2 -> 64 (rep_k=2, h_single=5.831Å, slab_length=10)
  - atoms_est hkl [-1  0 -1] sigma 2 -> 96 (rep_k=1, h_single=14.218Å, slab_length=10)
  - atoms_est hkl [-1  0 -1] sigma 2 -> 96 (rep_k=1, h_single=14.218Å, slab_length=10)
  - atoms_est hkl [-1  0  1] sigma 2 -> 128 (rep_k=2, h_single=8.223Å, slab_length=10)
  - atoms_est hkl [-1  0  1

In [8]:
search_results[-2]

{'hkl': array([ 0, -1, -1]),
 'sigma': 3,
 'strain': 0.0,
 'rotation_matrix': array([[-0.96072707, -0.24447554, -0.13128297],
        [-0.24447554,  0.52186967,  0.81724155],
        [-0.13128297,  0.81724155, -0.56114261]]),
 'axis_cart': array([-0.14013018,  0.87231579,  0.46843217]),
 'type': 'Crystal Axis/Normal'}

In [6]:
from sevenn.sevennet_calculator import SevenNetCalculator
calc = SevenNetCalculator(model = '/Users/jason/Desktop/LiNO2/checkpoint_sevennet_mf_ompa.pth',\
                          modal = 'omat24')
atoms = stct.to_ase_atoms()
atoms.calc = calc
bulk_energy_per_atom = atoms.get_potential_energy()/len(stct)

In [9]:
from jobflow import Flow
from jobflow.managers.local import run_locally
from interfacemaster.twinning_jobflow import TwinningJobflowMaker
from interfacemaster.twinning_search import search_low_index_twinning
from pymatgen.core.structure import Structure

maker = TwinningJobflowMaker(
    crystal_structure=stct,
    search_result=search_results[-2],
    calc_type="sevennet",
    ml_model_path="/Users/jason/Desktop/LiNO2/checkpoint_sevennet_mf_ompa.pth",
    calc_kwargs={"modal": "omat24", "device": "cpu"},
    bulk_energy_per_atom = bulk_energy_per_atom,
    termination_ftol = 0.15,
    termination_tol = 0.15,
    trials = 20,
    slab_length = 10,
)

# 3) 生成 job 和 flow
job = maker.make()
flow = Flow([job])

# 4) 本地运行
response = run_locally(flow, create_folders = True)

2026-02-15 09:05:46,962 INFO Started executing jobs locally
2026-02-15 09:05:46,964 INFO Starting job - Twinning GB BO-Relax (Symmetric Terminations) (8b90ef18-3395-41ba-932d-f2df2ec70d64)
所有端面的 label_termination:
  shift=0.2574 | top=LiNi_Immm_12 | bottom=O2_Immm_6 | signature=('LiNi_Immm_12', 'O2_Immm_6')
  shift=0.7574 | top=O2_Immm_6 | bottom=LiNi_Immm_12 | signature=('LiNi_Immm_12', 'O2_Immm_6')
已保存 2 个端面 slab 到: twinning_jobflow_results/termination_slabs
共找到 1 对等价端面，开始逐个优化...
  - 优化 termination pair=(0.2574, 0.2574) (n_calls=20)
最低界面能: 0.8699 J/m^2，优化所有 1 个端面...
  - 优化端面 1/1: dp1=0.2574, dp2=0.2574


/Users/jason/Documents/GitHub/interface_master/interfacemaster/twinning_jobflow.py:850: FutureWarning: Import ExpCellFilter from ase.filters
  ecf = ExpCellFilter(atoms, mask=[True, False, False, False, False, False])


所有流程已完成。
2026-02-15 09:08:43,852 INFO Finished job - Twinning GB BO-Relax (Symmetric Terminations) (8b90ef18-3395-41ba-932d-f2df2ec70d64)
2026-02-15 09:08:43,853 INFO Finished executing jobs locally


In [4]:
maker = TwinningJobflowMaker(
    crystal_structure=stct,
    search_result=search_results[6],
    calc_type="sevennet",
    ml_model_path="/Users/jason/Desktop/LiNO2/checkpoint_sevennet_mf_ompa.pth",
    calc_kwargs={"modal": "omat24", "device": "cpu"},
    bulk_energy_per_atom = bulk_energy_per_atom,
    termination_ftol = 0.15,
    termination_tol = 0.15,
    trials = 10,
    slab_length = 10,
)

# 3) 生成 job 和 flow
job = maker.make()
flow = Flow([job])

# 4) 本地运行
response = run_locally(flow, create_folders = True)

2026-02-13 22:26:22,940 INFO Started executing jobs locally
2026-02-13 22:26:22,943 INFO Starting job - Twinning GB BO-Relax (Symmetric Terminations) (0cf88dcb-8aad-4d54-92a5-4ca21b800165)
所有端面的 label_termination:
  shift=0.1103 | top=LiNiO2_P2/m_40 | bottom=LiNiO2_P2/m_40 | signature=('LiNiO2_P2/m_40', 'LiNiO2_P2/m_40')
  shift=0.3603 | top=LiNiO2_P2/m_40 | bottom=LiNiO2_P2/m_40 | signature=('LiNiO2_P2/m_40', 'LiNiO2_P2/m_40')
  shift=0.6103 | top=LiNiO2_P2/m_40 | bottom=LiNiO2_P2/m_40 | signature=('LiNiO2_P2/m_40', 'LiNiO2_P2/m_40')
  shift=0.8603 | top=LiNiO2_P2/m_40 | bottom=LiNiO2_P2/m_40 | signature=('LiNiO2_P2/m_40', 'LiNiO2_P2/m_40')
已保存 4 个端面 slab 到: twinning_jobflow_results/termination_slabs
共找到 1 对等价端面，开始逐个优化...
  - 优化 termination pair=(0.1103, 0.1103) (n_calls=10)
最低界面能: 1.5836 J/m^2，优化所有 1 个端面...
  - 优化端面 1/1: dp1=0.1103, dp2=0.1103


/Users/jason/Documents/GitHub/interface_master/interfacemaster/twinning_jobflow.py:518: FutureWarning: Import ExpCellFilter from ase.filters
  ecf = ExpCellFilter(atoms, mask=[True, False, False, False, False, False])


所有流程已完成。
2026-02-13 22:30:06,239 INFO Finished job - Twinning GB BO-Relax (Symmetric Terminations) (0cf88dcb-8aad-4d54-92a5-4ca21b800165)
2026-02-13 22:30:06,239 INFO Finished executing jobs locally
